# Notebook 04 – Temporal Analysis
## Echo Chambers in Climate Change Discussions on Reddit

This is the **core analysis notebook**. It computes all SNA metrics across time
and compares networks before/after major climate events.

### Analysis pipeline:
1. Load monthly graphs from disk
2. Run Louvain + compute all metrics for every month
3. Run event-based before/after comparison
4. Identify bridge users per period
5. Find persistent bridge users across all periods
6. Save all results to CSV for reporting

In [ ]:
import sys, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

from src.metrics import GraphMetrics
from src.visualizer import Visualizer

GRAPH_DIR   = Path('../data/processed/graphs')
FIGURES_DIR = Path('../figures')
REPORT_DIR  = Path('../report')
FIGURES_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

viz = Visualizer(output_dir=str(FIGURES_DIR))
gm  = GraphMetrics(betweenness_k=100, top_users=50)

print("✅ Imports successful")

## Step 1: Load Graphs

In [ ]:
with open(GRAPH_DIR / 'monthly_graphs.pkl', 'rb') as f:
    monthly_graphs = pickle.load(f)

with open(GRAPH_DIR / 'event_graphs.pkl', 'rb') as f:
    event_graphs = pickle.load(f)

print(f"Monthly graphs loaded : {len(monthly_graphs)} periods")
print(f"Event graphs loaded   : {len(event_graphs)} events")
print(f"Periods: {sorted(monthly_graphs.keys())}")

## Step 2: Compute Monthly Metrics (Louvain + All SNA metrics)

> ⏱️ This may take **5–20 minutes** depending on graph sizes.

In [ ]:
print("Computing metrics for all monthly graphs...")
temporal_df = gm.compute_temporal(monthly_graphs)

print(f"\n✅ Done! Shape: {temporal_df.shape}")
print(temporal_df[['num_nodes','num_edges','modularity','num_communities',
                    'degree_assortativity','avg_clustering']].to_string())

In [ ]:
# Save temporal metrics
temporal_df.to_csv(REPORT_DIR / 'temporal_metrics.csv')
print(f"✅ Saved → {REPORT_DIR / 'temporal_metrics.csv'}")
temporal_df.describe()

## Step 3: Visualise Temporal Trends

In [ ]:
# Main 4-panel temporal plot
viz.plot_temporal_metrics(temporal_df, filename='temporal_metrics.png')
plt.show()

In [ ]:
# ── Focus on Modularity alone (for the report) ──
fig, ax = plt.subplots(figsize=(16, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')

x = range(len(temporal_df))
y = temporal_df['modularity'].values
labels = list(temporal_df.index)

ax.plot(x, y, color='#00e5ff', linewidth=2.5, marker='o', markersize=4)
ax.fill_between(x, y, alpha=0.12, color='#00e5ff')

# Mark events on the plot
from src.network_builder import CLIMATE_EVENTS
import pandas as pd
for event_name, event_date in CLIMATE_EVENTS.items():
    event_period = str(pd.to_datetime(event_date).to_period('M'))
    if event_period in labels:
        xi = labels.index(event_period)
        ax.axvline(xi, color='#ff6b6b', linestyle='--', alpha=0.7, linewidth=1)
        ax.text(xi + 0.2, max(y)*0.95, event_name.replace('_', '\n'),
                fontsize=6, color='#ff6b6b', rotation=45)

step = max(1, len(labels) // 12)
ax.set_xticks(range(0, len(labels), step))
ax.set_xticklabels(labels[::step], rotation=45, ha='right', fontsize=8, color='#aaa')
ax.set_title('Network Modularity Over Time (with Climate Events)', color='white', fontsize=14)
ax.set_ylabel('Modularity', color='#aaa')
ax.set_xlabel('Time Period', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(color='#2d2d3a', linestyle='--', alpha=0.5)
for spine in ax.spines.values(): spine.set_color('#444')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'modularity_with_events.png'), dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

In [ ]:
# ── Assortativity focus ──
fig, ax = plt.subplots(figsize=(16, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')

y2 = temporal_df['degree_assortativity'].values
ax.plot(x, y2, color='#ff6b6b', linewidth=2.5, marker='s', markersize=4)
ax.fill_between(x, y2, alpha=0.12, color='#ff6b6b')
ax.axhline(0, color='#aaa', linewidth=1, linestyle=':')

ax.set_xticks(range(0, len(labels), step))
ax.set_xticklabels(labels[::step], rotation=45, ha='right', fontsize=8, color='#aaa')
ax.set_title('Degree Assortativity Over Time', color='white', fontsize=14)
ax.set_ylabel('Assortativity', color='#aaa')
ax.set_xlabel('Time Period', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(color='#2d2d3a', linestyle='--', alpha=0.5)
for spine in ax.spines.values(): spine.set_color('#444')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'assortativity_over_time.png'), dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## Step 4: Event-Based Before/After Comparison

In [ ]:
print("Computing event comparison metrics...")
event_df = gm.compare_events(event_graphs)

print("\nEvent comparison results:")
print(event_df[['before_modularity','after_modularity','delta_modularity',
                 'before_degree_assortativity','after_degree_assortativity']].to_string())

In [ ]:
event_df.to_csv(REPORT_DIR / 'event_comparison.csv')
print(f"✅ Saved → {REPORT_DIR / 'event_comparison.csv'}")

In [ ]:
# Plot event comparison for modularity
viz.plot_event_comparison(event_df, metric='modularity', filename='event_comparison_modularity.png')
plt.show()

# Plot for assortativity
viz.plot_event_comparison(event_df, metric='degree_assortativity',
                          filename='event_comparison_assortativity.png')
plt.show()

## Step 5: Bridge User Analysis

In [ ]:
# Bridge users for the most recent period
latest_label = sorted(monthly_graphs.keys())[-1]
G_latest = monthly_graphs[latest_label]

stats_latest = gm.compute_all(G_latest, label=latest_label)
partition_latest = stats_latest.get('_partition', {})

bridge_df = gm.find_bridge_users(G_latest, partition_latest, top_n=50)
print(f"Bridge users found (period {latest_label}): {len(bridge_df)}")
bridge_df.head(20)

In [ ]:
bridge_df.to_csv(REPORT_DIR / f'bridge_users_{latest_label}.csv', index=False)
viz.plot_bridge_users(bridge_df, top_n=20, filename='bridge_users_latest.png',
                      title=f'Top Bridge Users – {latest_label}')
plt.show()

In [ ]:
# Persistent bridge users across all periods
print("Finding persistent bridge users (this takes a while)...")
persistent = gm.find_persistent_bridges(monthly_graphs, min_periods=3, top_n=20)
print(f"\nPersistent bridge users (appear as bridges in ≥3 months): {len(persistent)}")
print(persistent.to_string())

persistent.to_csv(REPORT_DIR / 'persistent_bridge_users.csv', index=False)

## Step 6: Metric Correlation Matrix

In [ ]:
viz.plot_correlation_matrix(temporal_df, filename='metric_correlation.png')
plt.show()

## Step 7: Summary Statistics Table

In [ ]:
summary = temporal_df[['modularity','num_communities','degree_assortativity',
                        'avg_clustering','density','mean_betweenness']].describe()
print("\n=== SUMMARY STATISTICS ACROSS ALL TIME PERIODS ===")
print(summary.round(4).to_string())
summary.to_csv(REPORT_DIR / 'summary_statistics.csv')
print(f"\n✅ Saved → {REPORT_DIR / 'summary_statistics.csv'}")

## ✅ Done!
All metrics computed and saved. Proceed to **`05_visualization.ipynb`** for final publication-quality figures.